# Baomai R44 — inspect the whole frame and its physical boundaries

**Optional standard-CPU viewer.** Run all cells to verify and open the delivered review. No GPU, new inference or model training runs here. You can also download `review.html` directly and open it offline.

- All **507 current fronts** in 12 original photographs were inspected; all **508 stable identities**, including the earlier deletion, remain visible.
- R44 changes **100 rows, 35 corner locations and 122 box boundaries**. Unchanged rows stay on the same frame page.
- Switch R42/R43 history beside the latest R44 labels. Previous/Next changes photographs, not individual rows.
- A manually traced outline for **frame_0160 source row 2** shows the interpreted front contour. It is a review aid, not a segmentation training mask.
- **328 source rows have unresolved fields. Only frame_0383 is a complete assistant-reviewed candidate: 43 fronts in one photograph.** Eleven whole images remain held; no training split has been assigned.

PR131 returned the identical R43 review package, not a trained checkpoint. The additional320 upstream `frame_*` photographs have not entered the documented137-photo training lineage. This notebook does not change that fact. **Do not create another return PR merely to return an unchanged copy of this package.** New physical feedback is the useful next input.


In [ ]:
from pathlib import Path
import base64, hashlib, html, json, subprocess, sys, tempfile, urllib.request, zipfile

# Immutable Git object and content hashes. A branch moving cannot substitute a different capsule.
REPOSITORY = "Siuuuers/dl"
ARCHIVE_BLOB = "c93f72f95d4b323c8b085a4994b363d862d5a788"
ARCHIVE_SHA256 = "100bfaceb9281bcf8c37c48eaaacf5159df14b0d04dd0ec37d5dcd4fc89c672d"
ARCHIVE_BYTES = 6599840
MANIFEST_SHA256 = "6a52279ac5c53452cf7e6a3dbd6ea8f76d4aeae287bd2d2a23ae5b61a4c231bf"
LOCAL_REVIEW_ARCHIVE = ""  # Optional: an already downloaded R44 review.zip, for offline use.
DOWNLOAD_BUNDLE = False

work = Path(tempfile.mkdtemp(prefix="baomai_r44_"))
archive_path = work / "baomai_front_review_r44.zip"
if LOCAL_REVIEW_ARCHIVE:
    content = Path(LOCAL_REVIEW_ARCHIVE).read_bytes()
else:
    url = f"https://api.github.com/repos/{REPOSITORY}/git/blobs/{ARCHIVE_BLOB}"
    request = urllib.request.Request(url, headers={"Accept":"application/vnd.github+json", "User-Agent":"baomai-r44-review"})
    with urllib.request.urlopen(request, timeout=90) as response:
        result = json.load(response)
    assert result["sha"] == ARCHIVE_BLOB and result["encoding"] == "base64", "Unexpected Git object"
    content = base64.b64decode(result["content"])
assert len(content) == ARCHIVE_BYTES and hashlib.sha256(content).hexdigest() == ARCHIVE_SHA256, "Capsule identity mismatch"
archive_path.write_bytes(content)
folder = work / "review"
with zipfile.ZipFile(archive_path) as z:
    items = z.infolist()
    names = [x.filename for x in items]
    assert len(names) == len(set(names)), "Duplicate ZIP member"
    assert sum(x.file_size for x in items) < 100_000_000, "Unexpected expanded size"
    assert z.testzip() is None, "CRC failure"
    inventory = json.loads(z.read("inventory.json"))
    assert set(names) == set(inventory) | {"inventory.json"}, "Inventory mismatch"
    for item in items:
        rel = Path(item.filename)
        assert not rel.is_absolute() and ".." not in rel.parts and "\\" not in item.filename
        assert not item.is_dir() and ((item.external_attr >> 16) & 0o170000) != 0o120000
        payload = z.read(item.filename)
        if item.filename in inventory:
            assert inventory[item.filename] == {"sha256":hashlib.sha256(payload).hexdigest(), "bytes":len(payload)}, item.filename
        target = folder / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
assert hashlib.sha256((folder / "manifest.json").read_bytes()).hexdigest() == MANIFEST_SHA256
checked = subprocess.run([sys.executable, str(folder / "review.py"), "--check-capsule", str(archive_path)], check=True, capture_output=True, text=True)
verification = json.loads(checked.stdout)
print("Verified all 12 original images, 508 identities and 507 current fronts.")
print("This round:", verification["latest_counts"]["amended_rows"], "amended rows;", verification["latest_counts"]["retained_rows"], "reviewed and retained.")
print("Complete annotation candidates:", ", ".join(verification["annotation_complete_ids"]))
print("Held whole images:", ", ".join(verification["held_image_ids"]))
print("No training or model promotion occurred.")
print("Capsule:", archive_path)


In [ ]:
manifest = json.loads((folder / "manifest.json").read_text())
complete = [f for f in manifest["frames"] if f["eligible_for_candidate_export"]]
held = [f for f in manifest["frames"] if not f["eligible_for_candidate_export"]]
print("Complete candidate photos:", len(complete), "; fronts in those photos:", sum(f["candidate_rows"] for f in complete))
print("Held whole photos:", len(held), "; held source rows:", manifest["counts"]["held_source_rows"])
print("These are annotation candidates, not an assigned training set. No GPU experiment is started.")
print("First useful boundary clarification: frame_0160 source37 ymin; it is that photograph's only remaining hold.")
assert all(f["training_split"] == "unassigned" and not f["eligible_for_independent_evaluation"] for f in manifest["frames"])
assert not manifest["training_run"] and not manifest["active_dataset_modified"]


In [ ]:
from IPython.display import HTML, display
viewer = (folder / "review.html").read_text()
display(HTML('<iframe title="R44 original-pixel review" sandbox="allow-scripts" style="width:100%;height:1100px;border:0" srcdoc="' + html.escape(viewer, quote=True) + '"></iframe>'))
if DOWNLOAD_BUNDLE:
    try:
        from google.colab import files
    except ImportError:
        print("Open the capsule path printed above; review.html is self-contained after extraction.")
    else:
        files.download(str(archive_path))


## Read the changes, then clarify the uncertain material

Start with **All rows** and select `frame_0160`. Jump to source row2, enable the review outline, and compare the old left point against the visible sloped side. The latest nominal left point is **(373,309)** in the original1280×720 image. Green/blue identify physical left/right turns; magenta identifies a changed field. A point does not have to equal a rectangular box corner.

The original-pixel viewer preserves owned shaded rims. Gray beside a continuing upright can clearly be rack; gray between white bands may belong to a front or to a receding surface. HistoricalPR30 examples include both excluding rack and restoring gray front material omitted by a thin white-band box. Do not apply one brightness threshold or one fixed inset to all rows.

The traced polygon has approximately1.5px visual uncertainty. It exposes the current interpretation and is not an exact segmentation mask. Unavailable corners remain `0 0 0`; they receive no drawn point.

The highest-value next clarification is **frame_0160 source37's upper frontal edge**, its only remaining hold. A trace of that actual front/upper-plane junction could qualify the whole photo. **frame_0001 source11** is a second useful example of the recurring gray-interface ambiguity.

The five saved prediction comparisons remain historicalR42 figures. New labels alone do not demonstrate fewer misses, fewer duplicates or better model predictions. After a useful supplementary set is fully qualified, freeze its originals/labels/split, then compare the same baseline training with and without those additional photos. Match optimizer-update budgets or report the extra-exposure difference explicitly. Keep development references fixed; these mined review photos are not an independent test set.

`proposals/` includes held rows for review. Do not use that directory as an automatic training-label glob. The conservative whole-image export is under `complete-candidates/labels/` and still needs an explicit training manifest.

Set `DOWNLOAD_BUNDLE = True` in the first code cell only if you want Colab to download the portableZIP as well as show the viewer. No account token or API key is required for this public immutable artifact.
